In [0]:
import requests, json

meta = requests.get("https://data.opensanctions.org/datasets/latest/index.json").json()


url=meta["statements_url"]

SNAPSHOT_VERSION = url.rstrip("/").split("/")[-2]
print(SNAPSHOT_VERSION)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pep_dev.bronze.snapshot_version_control (
    snapshot_version STRING,
    created_at TIMESTAMP
)
USING DELTA;

--    INSERT INTO pep_dev.bronze.snapshot_version_control (snapshot_version, created_at)
--    SELECT DISTINCT _snapshot_version, current_timestamp()
--    FROM pep_dev.bronze.statements;

In [0]:
# dbutils.widgets.remove("SNAPSHOT_VERSION")
dbutils.widgets.text("SNAPSHOT_VERSION", SNAPSHOT_VERSION)

In [0]:
# target = dbutils.widgets.get("SNAPSHOT_VERSION")

n = spark.sql(f"""
  SELECT count(*) AS n FROM pep_dev.bronze.snapshot_version_control
  WHERE snapshot_version = :v
""", args={"v":SNAPSHOT_VERSION}).collect()[0]["n"]


In [0]:

dbutils.jobs.taskValues.set(
    key="SNAPSHOT_ROW_CONT",
    value=n
)

dbutils.jobs.taskValues.set(
    key="SNAPSHOT_VERSION",
    value=dbutils.widgets.get("SNAPSHOT_VERSION")
)

dbutils.jobs.taskValues.set(
    key="source_path",
    value=url
)

In [0]:
# n=spark.sql(f"""SELECT count(*) AS n from pep_dev.bronze.statements
# WHERE _snapshot_version = '{SNAPSHOT_VERSION}'""").collect()[0]["n"]
# display(n)

Built a  snapshot version control to make snapshot versions accessible across master jobs as job level boundaries didn't allow that